# 07 — Đánh giá Retrieval và Câu trả lời (Evaluation)

Phase 7 đánh giá chất lượng của hệ thống Hue Foods RAG theo luồng trực quan:
```text
question -> retrieve -> build context -> generate -> judge -> report
```

- **Retrieval Evaluation**: Đánh giá khả năng tìm đúng tài liệu chứa từ khóa mong đợi (`MRR`, `nDCG`, `Keyword Coverage`).
- **Answer Evaluation**: Sử dụng `gpt-5.4-nano` để sinh câu trả lời và `gpt-5.4-mini` làm giám khảo (LLM-as-a-judge) chấm 3 tiêu chí: `accuracy`, `completeness`, `relevance` (thang điểm 1–5) cùng `feedback`.

## 1. Thiết lập đường dẫn backend

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Không tìm thấy thư mục backend/. Hãy mở notebook từ repo root hoặc thư mục notebooks/."
    )
print(f"backend on path: {sys.path[0]}")

## 2. Import các hàm cần dùng

In [ ]:
import asyncio
from evaluation.test import DEFAULT_TEST_FILE, load_tests
from evaluation.eval import (
    build_services,
    evaluate_answer,
    evaluate_retrieval,
    run_answer_batch,
    run_retrieval_batch,
)
from evaluation.evaluator import build_app

## 3. Đọc 20 câu hỏi thật từ test2.jsonl

In [ ]:
questions = load_tests(DEFAULT_TEST_FILE)
print(f"Tổng số câu hỏi: {len(questions)}")

## 4. Xem một câu hỏi mẫu

In [ ]:
first_question = questions[0]
first_question

## 5. Chạy retrieval cho một câu hỏi

In [ ]:
services = build_services("dense_only")
retrieval_result = evaluate_retrieval(first_question, services)
retrieval_result

## 6. Ý nghĩa của MRR và nDCG

- **MRR (Mean Reciprocal Rank)**: Đánh giá vị trí xuất hiện đầu tiên của từ khóa trong danh sách kết quả ($1/\text{rank}$).
- **nDCG (Normalized Discounted Cumulative Gain)**: Đánh giá chất lượng xếp hạng tổng thể của các đoạn văn chứa từ khóa.
- **Keyword Coverage**: Tỷ lệ phần trăm từ khóa quan trọng được tìm thấy trong top kết quả.

## 7. Sinh và chấm một câu trả lời thật

Hệ thống truy xuất ngữ cảnh, sinh câu trả lời bằng `gpt-5.4-nano` và chấm điểm bằng `gpt-5.4-mini`. Từ khóa `await` cho phép chờ kết quả từ API trực tuyến mà không chặn các tiến trình khác.

In [ ]:
answer_result = await evaluate_answer(first_question, services)
answer_result

## 8. Xem ba điểm chất lượng và feedback

In [ ]:
print(f"Accuracy: {answer_result['accuracy']}/5")
print(f"Completeness: {answer_result['completeness']}/5")
print(f"Relevance: {answer_result['relevance']}/5")
print(f"Feedback: {answer_result['feedback']}")

## 9. Chạy retrieval cho toàn bộ 20 câu hỏi

In [ ]:
retrieval_rows, retrieval_summary = run_retrieval_batch(DEFAULT_TEST_FILE, 3)
retrieval_summary

## 10. Chạy answer evaluation cho toàn bộ 20 câu hỏi

In [ ]:
answer_rows, answer_summary = await run_answer_batch(DEFAULT_TEST_FILE, 3)
answer_summary

## 11. Giao diện Gradio

Để khởi chạy giao diện web đánh giá trực quan, gọi `build_app().launch(inbrowser=True)`.

In [ ]:
app = build_app()
app